# 01 — Configuration and Data Loading

Load paths and analysis parameters from `config.yaml` (formal) or `config.example.yaml` (sample), then read the HMM model, parcel time series, ROI mask, and Schaefer atlas labels.

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

PROJECT_ROOT = Path('..').resolve()
CONFIG_NAME = os.environ.get('GRAPH_THEORY_CONFIG', 'config.yaml')
CONFIG_PATH = PROJECT_ROOT / CONFIG_NAME
if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f'{CONFIG_NAME} not found. Use config.example.yaml (sample) or config.yaml (formal).'
    )

with open(CONFIG_PATH, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

use_example_data = bool(cfg.get('use_example_data', False))
paths = cfg['paths']
analysis = cfg['analysis']
graph_cfg = cfg['graph_theory']

K = int(analysis['K'])
method = analysis['method']
n_timepoints = int(analysis['n_timepoints'])
n_subjects = int(analysis['n_subjects'])
use_gamma = bool(analysis['use_gamma'])

DATA_DIR = (PROJECT_ROOT / paths['data_dir']).resolve()
HMM_DIR = (PROJECT_ROOT / paths['hmm_models_dir']).resolve()
RESULTS_DIR = (PROJECT_ROOT / paths['results_dir']).resolve()
ROI_MASK = (PROJECT_ROOT / paths['roi_mask']).resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Configuration loaded.')
print(f'  use_example_data={use_example_data}')
print(f'  parcels=1000, K={K}, method={method}')
print(f'  data_dir={DATA_DIR}')


Configuration loaded.
  use_example_data=True
  parcels=1000, K=3, method=example
  data_dir=D:\文件同步\文件\数据分析结果\data_code_figure\adaptive_reserve_moderate\graph_theory\example_data


In [2]:
if use_example_data:
    labels_df = pd.read_csv(DATA_DIR / 'network_labels.csv')
    select_networks = labels_df['network'].values
    subjects = pd.read_csv(DATA_DIR / 'subject_indices.csv')['subject'].tolist()
    hmm = None
    print('Example mode: skipped HMM/BIDS loading.')
    print(f'Example subjects: {subjects}')
else:
    import pickle
    from bids import BIDSLayout
    from glhmm import statistics

    bids_root = PROJECT_ROOT / paths['project_root']
    layout = BIDSLayout(bids_root, validate=False, derivatives=True)
    subjects = layout.get_subjects()
    indices = statistics.get_indices_timestamp(n_timepoints, n_subjects)

    hmm_path = HMM_DIR / f'trained_hmm_1000parcels_{K}states_{method}.pkl'
    with open(hmm_path, 'rb') as f:
        hmm = pickle.load(f)

    print(f'Loaded HMM: {hmm_path.name}')
    print(f'Subjects (BIDS): {len(subjects)}')


Example mode: skipped HMM/BIDS loading.
Example subjects: ['sub-01', 'sub-02', 'sub-03']


In [3]:
if use_example_data:
    sort_idx = np.argsort(select_networks)
    sorted_networks = select_networks[sort_idx]
    select_networks = sorted_networks

    roi_dir = RESULTS_DIR / '01_roi'
    roi_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({'network': sorted_networks}).to_csv(roi_dir / 'sorted_networks.csv', index=False)
    np.save(roi_dir / 'sort_idx.npy', sort_idx)

    Y = pd.read_csv(DATA_DIR / 'parcel_timeseries.csv').values[:, sort_idx]
    print(f'Signal shape: {Y.shape}')
else:
    def read_parcel_data(method):
        signal_file = DATA_DIR / f'all_schaefer_parcel1000_{method}.csv'
        indices_file = DATA_DIR / f'all_schaefer_indices_parcel1000_{method}.csv'
        all_schaefer = pd.read_csv(signal_file).values
        all_schaefer_indices = pd.read_csv(indices_file).values
        print(f'Signal shape: {all_schaefer.shape}')
        print(f'Indices shape: {all_schaefer_indices.shape}')
        return all_schaefer, all_schaefer_indices

    all_schaefer, all_schaefer_indices = read_parcel_data(method=method)
    Y = all_schaefer


Signal shape: (360, 18)


In [4]:
if not use_example_data:
    import nibabel as nib
    from nilearn.image import math_img

    schaefer_dir = paths.get('schaefer_dir')
    if schaefer_dir:
        schaefer_base = Path(schaefer_dir).expanduser()
    else:
        schaefer_base = Path.home() / 'nilearn_data' / 'schaefer_2018'

    schaefer_img_path = schaefer_base / (
        'Schaefer2018_1000Parcels_7Networks_order_FSLMNI152_2mm.nii.gz'
    )
    label_file = schaefer_base / 'Schaefer2018_1000Parcels_7Networks_order.txt'

    merged_stats = nib.load(str(ROI_MASK))
    schaefer_data = nib.load(str(schaefer_img_path))
    inter_img = math_img('img1 * img2', img1=merged_stats, img2=schaefer_data)

    labels = pd.read_csv(label_file, sep='\t', header=None)
    networks = labels[1].str.split('_', expand=True)[2]
    network_names = networks.values

    inter_data = inter_img.get_fdata()
    unique_labels = np.unique(inter_data[inter_data > 0])
    print(f'ROI labels after mask intersection: {len(unique_labels)}')

    select_networks = [network_names[int(label) - 1] for label in unique_labels]

    unique_networks, network_counts = np.unique(select_networks, return_counts=True)
    valid_networks = unique_networks[network_counts >= 3]
    network_mask = pd.Series(select_networks).isin(valid_networks)

    unique_labels = unique_labels[network_mask.to_numpy()]
    select_networks = np.array(select_networks)[network_mask.to_numpy()]
    Y = Y[:, network_mask.to_numpy()]

    sort_idx = np.argsort(select_networks)
    sorted_networks = select_networks[sort_idx]
    select_networks = sorted_networks
    Y = Y[:, sort_idx]

    roi_dir = RESULTS_DIR / '01_roi'
    roi_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({'network': sorted_networks}).to_csv(roi_dir / 'sorted_networks.csv', index=False)
    np.save(roi_dir / 'sort_idx.npy', sort_idx)
else:
    print(f'Example mode: using packaged ROI network labels ({len(select_networks)} ROIs).')

select_networks = np.asarray(select_networks)
print(f'Retained ROIs: {Y.shape[1]}')
print(f'Networks: {sorted(set(select_networks))}')


Example mode: using packaged ROI network labels (18 ROIs).
Retained ROIs: 18
Networks: ['SalVentAttn', 'SomMot', 'Vis']


In [5]:
# Save shared metadata for downstream notebooks
meta = {
    'use_example_data': use_example_data,
    'K': K,
    'method': method,
    'n_timepoints': n_timepoints,
    'n_subjects': n_subjects,
    'use_gamma': use_gamma,
    'subjects': subjects,
    'select_networks': select_networks.tolist(),
}
meta_path = RESULTS_DIR / 'analysis_metadata.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print(f'Saved metadata: {meta_path}')


Saved metadata: D:\文件同步\文件\数据分析结果\data_code_figure\adaptive_reserve_moderate\graph_theory\results\example\analysis_metadata.json
